# Module 04 — Lifecycle Hooks

Hooks let you run custom code at specific points in the agent's lifecycle — without modifying the agent's core logic.

### Available hook events

| Event | When it fires |
|---|---|
| `AgentInitializedEvent` | Once, right after the agent is created |
| `MessageAddedEvent` | Every time a message is added to the conversation |
| `AfterInvocationEvent` | After each call to `agent(...)` completes |

### Why hooks?

Hooks keep concerns separated:
- **Logging** — track every message without cluttering agent code
- **Memory** — load preferences on init, save conversations after each turn
- **Guardrails** — inspect messages before they're processed
- **Analytics** — count tokens, measure latency, track tool usage

In [ ]:
from strands import Agent
from strands.hooks import (
    AgentInitializedEvent,
    MessageAddedEvent,
    AfterInvocationEvent,
    HookProvider,
    HookRegistry
)

### Simple logging hook

The simplest hook just logs what's happening.

In [ ]:
class LoggingHook(HookProvider):
    """Logs agent lifecycle events to the console."""
    
    def on_agent_initialized(self, event: AgentInitializedEvent):
        print(f"[HOOK] Agent '{event.agent.name}' initialized")
    
    def on_message_added(self, event: MessageAddedEvent):
        role = event.message.get('role', 'unknown')
        content = event.message.get('content', [])
        text = content[0].get('text', '')[:60] if content else ''
        print(f"[HOOK] Message added — role={role}: {text!r}")
    
    def on_after_invocation(self, event: AfterInvocationEvent):
        msg_count = len(event.agent.messages)
        print(f"[HOOK] Invocation complete — {msg_count} messages in history")
    
    def register_hooks(self, registry: HookRegistry):
        registry.add_callback(AgentInitializedEvent, self.on_agent_initialized)
        registry.add_callback(MessageAddedEvent, self.on_message_added)
        registry.add_callback(AfterInvocationEvent, self.on_after_invocation)


agent = Agent(
    name="LoggedAgent",
    model="us.anthropic.claude-3-5-haiku-20241022-v1:0",
    hooks=[LoggingHook()]
)

agent("What is the capital of India?")

### Context injection hook

A common pattern: use `AgentInitializedEvent` to inject context into the system prompt before the first message.

In [ ]:
class ContextInjectionHook(HookProvider):
    """Injects user context into the system prompt on agent init."""
    
    def __init__(self, user_context: str):
        self.user_context = user_context
    
    def on_agent_initialized(self, event: AgentInitializedEvent):
        # Append context to the existing system prompt
        event.agent.system_prompt += f"\n\n## User Context:\n{self.user_context}"
        print(f"[HOOK] Injected context into system prompt")
    
    def register_hooks(self, registry: HookRegistry):
        registry.add_callback(AgentInitializedEvent, self.on_agent_initialized)


# Simulate context that might come from a database or memory store
user_context = """- Name: Yeshwanth
- Dietary restrictions: vegetarian
- Favorite cuisine: Japanese
- Dislikes: cilantro"""

agent = Agent(
    name="PersonalChef",
    model="us.anthropic.claude-3-5-haiku-20241022-v1:0",
    system_prompt="You are a personal food assistant. Be concise.",
    hooks=[ContextInjectionHook(user_context)]
)

agent("Recommend a dish for me tonight.")

### Conversation saver hook

Use `AfterInvocationEvent` to persist conversations — this is the foundation of memory.

In [ ]:
class ConversationSaverHook(HookProvider):
    """Saves conversation turns to a local list (simulating a database)."""
    
    def __init__(self):
        self.saved_turns = []
    
    def on_after_invocation(self, event: AfterInvocationEvent):
        messages = event.agent.messages
        if len(messages) < 2:
            return
        
        # Get the last user message and assistant response
        user_msg = None
        assistant_msg = None
        
        for msg in reversed(messages):
            if msg['role'] == 'assistant' and not assistant_msg:
                content = msg.get('content', [])
                if content and isinstance(content[0], dict):
                    assistant_msg = content[0].get('text', '')
            elif msg['role'] == 'user' and not user_msg:
                content = msg.get('content', [])
                if content and isinstance(content[0], dict):
                    user_msg = content[0].get('text', '')
                    break
        
        if user_msg and assistant_msg:
            self.saved_turns.append({"user": user_msg, "assistant": assistant_msg})
            print(f"[HOOK] Saved turn #{len(self.saved_turns)} to memory")
    
    def register_hooks(self, registry: HookRegistry):
        registry.add_callback(AfterInvocationEvent, self.on_after_invocation)


saver = ConversationSaverHook()

agent = Agent(
    model="us.anthropic.claude-3-5-haiku-20241022-v1:0",
    system_prompt="You are a helpful assistant. Be brief.",
    hooks=[saver]
)

agent("I love Thai food.")
agent("I'm allergic to Non-veg.")

print("\nSaved conversation turns:")
for i, turn in enumerate(saver.saved_turns, 1):
    print(f"\nTurn {i}:")
    print(f"  User     : {turn['user']}")
    print(f"  Assistant: {turn['assistant'][:80]}...")

### Combining multiple hooks

Pass a list of hook providers — they all fire in order.

In [ ]:
agent = Agent(
    name="MultiHookAgent",
    model="us.anthropic.claude-3-5-haiku-20241022-v1:0",
    system_prompt="You are a helpful assistant.",
    hooks=[
        LoggingHook(),
        ContextInjectionHook("User prefers short answers."),
        ConversationSaverHook()
    ]
)

agent("What's a good snack?")

---

### Key takeaways

- Implement `HookProvider` and override the event methods you need
- `register_hooks` wires your callbacks to the registry
- `AgentInitializedEvent` → inject context, load memory
- `AfterInvocationEvent` → save conversations, log metrics
- `MessageAddedEvent` → inspect/filter messages in real time
- Pass multiple hooks as a list — they compose cleanly

Next up: **05_memory.ipynb** — persistent memory with AWS AgentCore.